# Working with MLPs in pyMC

This notebook illustrates the definition of an MLP in PyMC by importing a [.pt file generated by pytorch](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html), as well as its integration within a DAG, and its evaluation and differentiation. 

We start by importing PyMC. We need to make sure we are using version 5 or higher and that it was compiled with Torch enabled.

In [1]:
import pymcpp

We instantiate an MLP and populate it with data read from the file `peak_ReLU_20L2.pt`. This file defines a simple neural network with two inputs, two hidden layers with 20 neurons in each layer, and a single output predicting the value of the Matlab's peak function. 

In [2]:
NN = pymcpp.MLP()
NN.read_data( "peak_ReLU_20L2.pt" )
NN.options.AUTODIFF = NN.options.F
NN.varout[0].str()

'(-0.51838886737823) + (-0.99422514438629) * MAX( (-0.39564689993858) + 0.79413586854935 * MAX( 0.39745235443115 + (-0.80860245227814) * X0 + 1.2753248214722 * X1, 0 ) + (-0.26635843515396) * MAX( (-0.27403348684311) + (-1.1127458810806) * X0 + (-0.83447587490082) * X1, 0 ) + 0.17339082062244 * MAX( 0.2478834092617 + (-0.88525533676147) * X0 + 0.21976538002491 * X1, 0 ) + 0.49911120533943 * MAX( (-1.0652762651443) + (-0.06457332521677) * X0 + 0.73023051023483 * X1, 0 ) + 0.13976338505745 * MAX( (-0.71780675649643) + (-0.14013853669167) * X0 + 0.47389468550682 * X1, 0 ) + (-0.15548038482666) * MAX( (-0.32799020409584) + (-0.87875580787659) * X0 + (-0.23392949998379) * X1, 0 ) + 0.21824967861176 * MAX( 0.98978573083878 + (-1.1221823692322) * X0 + (-0.062369730323553) * X1, 0 ) + 0.016908891499043 * MAX( 0.84667313098907 + (-0.41811281442642) * X0 + 0.41338017582893 * X1, 0 ) + 1.0134687423706 * MAX( (-0.87527066469193) + 0.49559560418129 * X0 + 0.71499013900757 * X1, 0 ) + 0.299077779054

Next, we instantiate a DAG, define a list of 2 DAG variables, instantiate an MLP operation, and tie this MLP operation to the network `NN` above. 

In [3]:
DAG  = pymcpp.FFGraph()
X    = DAG.add_vars( len(NN.varin) )
OpNN = pymcpp.FFMLP()
F    = OpNN( X, NN )
print( F[0].str() )

MLP[0x1862e6b0][0]( V0, V1 )


We can readily evaluate the expression `F`, here at the origin $(0,0)$.

In [4]:
F0 = DAG.eval( F, X, [0,0] )
print( F0 )

[0.9258726368242958]


We can also differentiate the expression `F` containing the MLP operation and evaluate the derivative expressions at $(0,0)$.

In [5]:
dFdX = DAG.fdiff( F, X )
print( [ dFdX[2][i].str() for i in range(len(dFdX[2])) ] )

dFdX0 = DAG.eval( dFdX[2], X, [0,0] )
print( [ dFdX0[i] for i in range(len(dFdX[2])) ] )

['GradMLP[0x18830170][0]( V0, V1 )', 'GradMLP[0x18830170][1]( V0, V1 )']
[-3.766655750066504, -1.259230185869002]


The evaluation can also be conducted in other arithmetics, for instance applying interval arithmetic in the small interval $[-0.01,0.01]^2$ around the origin.

In [6]:
IF0 = DAG.eval( F, X, [pymcpp.Interval(-0.01,0.01), pymcpp.Interval(-0.01,0.01)] )
print( IF0 )

IdFdX0 = DAG.eval( dFdX[2], X, [pymcpp.Interval(-0.01,0.01), pymcpp.Interval(-0.01,0.01)] )
print( IdFdX0 )

[[  6.7581040336198284e-01 :  1.1759348702866070e+00 ]]
[[ -3.7666557500665050e+00 : -3.7666557500665028e+00 ], [ -1.2592301858690034e+00 : -1.2592301858690009e+00 ]]


This example illustrates the large overestimation caused by the wrapping effect and dependency problem of interval arithmetic, even for a simple network.